JMG Please submit your notebooks with the output from the code cells showing.  No output, no credit.

## Part One:  Introduction

####  The model 

In the formulation of Chapter 8 of our text, an  attention **head** computes 
an output vector at position $i$ ($\mathbf{a_{i}}$) as 
a function of the input vector $\mathbf{x_{i}}$ and all the input vectors to the left of $\mathbf{x_{i}}$.
In particular, $\mathbf{a_{i}}$ is a weighted sum of the **value vectors** 
of the vectors $\mathbf{x_{1},\dots x_{i}}$, where the value
vector $\mathbf{v_{j}}$ is defined in terms of the data vector at
position $j$ (numbering follows the numbering in the text; an M means the equation has been
modified or expanded):

$$
(8.10)\quad\mathbf{v_{j}} = \mathbf{\text{x}_{i}} \text{W}^{V}. 
$$

The goal of this assignment is to take you through the nitty-gritty of that computation with
made-up data.  It presupposes
a thorough knowledge of Chapter 8 (especially 8.10-8.14).



We will write the **weight** for the value vector at position $j$ relative to position $i$ ( $j \leq i$ ) 
as $a_{ij}$:

$$
(8.11M\,8.12M) \quad a_{ij} = \text{softmax}_{i}\left (\, \frac{\mathbf{{q_{i}}} \cdot \mathbf{k_{j}}}{\sqrt{d_{k}}} \,\right)
= \text{softmax}_{i}\left ( \text{score}(\mathbf{x_{i},x_{j}}) \right)
%=  \frac{\text{exp}(\text{score}(X_{i},X_{j}))}{\sum_{j\leq i} \text{exp}(\text{score}(X_{i},X_{j}))}
$$

That is, do a softmax of the scaled dot products of each of the $x_{i}$ to the left of 
$x_{j}$ with $x_{j}$ .
Here we've followed the text and rewritten  the argument of softmax 

$$
\frac{\mathbf{q_{i}} \cdot \mathbf{k_{j}}}{\sqrt{d_{k}}}
%\frac{Xq_{i} \cdot Xk_{j}}{\sqrt{d_{k}}}, 
$$


as a function of the original data vectors $\mathbf{x_{i}}$ and $\mathbf{x_{j}}$:

$$
\text{score}(\mathbf{x_{i},x_{j}}) = \frac{\mathbf{q_{i}} \cdot \mathbf{k_{j}}}{\sqrt{d_{k}}}.
$$

Note that 

$$
(8.10)\quad\mathbf{{q_{i}}} = \mathbf{\text{x}_{i}} \text{W}^{Q} \quad \mathbf{k_{j}} = \mathbf{\text{x}_{j}}  \text{W}^{K}.
$$


Then substituting in the definition of softmax we have:

$$
(8.12M)\quad a_{ij} =  \text{softmax}_{i}\left ( \text{score}(\mathbf{x_{i},x_{j}}) \right)
=  \frac{\text{exp}(\text{score}(\mathbf{x_{i},x_{j}}))}{\sum_{j\leq i} \text{exp}(\text{score}(\mathbf{x_{i},x_{j}}))}
$$

Note that softmax isn't just a function of the scalar $\text{score}(\mathbf{x_{i},x_{j}})$;
you also need to specify $i$ in order to be able to sum all the exponentiated scores for vectors less than $i$.

The weighted sum of the value vectors referred
to above is the vector called $\mathbf{\text{head}}_{i}$ in the text: 

$$
(8.13)\quad \mathbf{\text{head}_{i}} = \sum_{j\leq i} \alpha_{ij} \mathbf{v_{j}}.
$$


The output vector of the attention module at position $i$ ($\mathbf{a_{i}}$) is the result of passing $\mathbf{\text{head}}_{i}$ through one more matrix, $\text{W}^{O}$, which projects 
$\mathbf{\text{head}}_{i}$ down to a vector that has the same dimensionality as the input:

$$
(8.14)\quad \mathbf{a_{i}} = \mathbf{\text{head}}_{i} \text{W}^{O}.
$$


#### The data, weight matrices, and output matrices

Imagine that these are the seven words of a  sentence

> A relative gave the baby the present.

that is passed as input to an attention module (so N=7).
The cell below titled **Part one Attention module** 
defines a matrix `X` with 7 rows 
containing made-up word vectors for this input
as well as the three **weight** matrices and one **output** matrix 
necessary for a single head attention module of the sort described in section
8.1 of the text and pictured in Figure 8.4.  The computation
has been parallelized as described in Section 8.3, but without
the complication of a multi-headed attention module.

Following the discussion in Chapter 8, 8.1-8.3,  the three weight matrices referred
to there as $\text{W}^{K}$,  $\text{W}^{Q}$,  $\text{W}^{V}$, and the output matrix $\text{W}^{O}$, 
are called, respectively, `Wk`, `Wq`, `Wv`, and `Wo` below.  Your task is to compute some of the output of the Part One attention module using the equations and description given in the text.  As your step-by-step guide, answer the questions below. One of the things you will have to compute is the NxN $QK^{T}$ matrix which has the query-value dot products shown in Figure 8.8.  

###  Part One Attention Module

In [361]:
import numpy as np
import random


np.random.seed(47)
shape = (7,4)
# Number of words in input
N = shape[0]
# X: Matrix of embeddings (word vectors) for input (same as X in text)
#    A matrix of values between 0 and 1 of shape shape[0] x shape[1]
X = np.random.rand(*shape)
## Sentence is "A relative gave the baby the present."
## Make the 6th word vector be the same as the 4th.
X[5,:] = X[3,:].copy()
Wk = np.random.rand(4,3)  
Wq = np.random.rand(4,3)   
Wv = np.random.rand(4,5)     
Wo =  np.random.rand(5,4) 

In [383]:
#Verifying the 4th and 6th word vecs are the same:
print(X[3])
print(X[5])

[0.09872595 0.30043644 0.64085568 0.32220795]
[0.09872595 0.30043644 0.64085568 0.32220795]


1.  Let's call the dimensionality of the input and output embeddings of the part One Attention module $d$; and let's call the dimensionality of the key vector $k_{j}$ $d_{k}$ (see Equation 8.10 in the text).  Similarly we'll call the dimensionality of $q_{i}$ $d_{q}$ and the dimensionality of $v_{j}$ $d_{v}$  (also Equation 8.10 in the text).
What are $d$, $d_{k}$, $d_{q}$, and $d_{v}$ in the Part One Attention model?   Finally, what is the shape of the  output vector $a_{i}$?

2.  In Section 8.3 the required query-value comparisons (dot products) are all computed in a single  matrix J&M call $QK^{T}$.  Compute that matrix following the description in the text and call it `QKt`.Also **scale** those dot products correctly (as described way back in equation 8.16) and put the result in another matrix called  `scores`.  Try to do that efficiently (i.e., not with a loop).   Note that the values in `scores` are the heart of the attention module.  They essentially tell us how much each word in context should be weighted in computing the output for the current word.

3. Validate your computation of  `scores` by looking up the query-value score for `X[3,:]` (as query) and `X[2,:]` (as key)  in `scores` and then computing it individually using Equation 8.16. The two computations should give the same result.
   
4. For this part we will compute `softmax` values using the values in `scores`.  Note: as the definition of softmax given above shows, softmax values must be computed relative to a position $i$ (you are summing over products computed for position $i$ and all positions to the left of $i$).  Use the matrix `scores` (computed in answering question 2), but be sure to exclude computations involving any vectors to the right of position $i$. For $i$, use $i=3$; with 0-based indexing, that means you will be computiong softmax relative to the 4th input vector $\mathbf{x_{3}}$.

5.  What is $\mathbf{\text{head}_{i}}$, the weighted sum vector for position $i$, and $a_{i}$, the vector output by the attention module for position $i$, when $i=3$? Use equations (8.13) and (8.14) but you will need to consider equations (8.10)-(8.14). 
 
6. What are $\mathbf{\text{head}_{i}}$ and $a_{i}$ when $i=5$? Note that since our input sentnce is *"The boy gave the baby the present*, position $i=5$ has the same word vector as position $i=3$. Is $a_{5}$ the same as $a_{3}$?  Why or why not? Is $q_{5}$ the same as $q_{3}$?  Why or why not?
   
7. Add another head to the model.  You will have to make up your own weight and output matrices for the second head. Use the same sentence and word vectors.  Use $\mathbf{\text{head}_{i}}$ unchanged.  Write whatever code you need to have your new attention head create another head vector $\mathbf{\text{head2}_{i}}$ 
and compute a new value for $a_{i}$ when $i=3$. You will need to use equation (8.19). It may help to  know authors J&M use $\mathbf{w\oplus c}$ to mean the concatenation of $\mathbf{w}$ and $\mathbf{c}$.  
So if $\mathbf{w}$ and $\mathbf{c}$ both have shape (4,), then $\mathbf{w\oplus c}$ has shape (8,).

8. Question 8 is found in part Two of the assignment..

####  Solution to question 1



####  Solution to questions 2 and  4 

In [387]:
###########   dk, Xk, Xq, Xv  #################

dk=3
Xk = ?
Xq = ?
Xv = ?

#################  Question 2 ############################################
# QKt[i,j] is the unscaled score for query vector i and key vector j
# jth row of QKt contains the N dot products scores for query vector j 
QKt = ?
scores = ?



#################  Question 4 ############################################
def compute_softmax (scores,i):
    # Softmaxing must be done wrt to a particular position i
    # Your code here
    pass

    
i = 3
#QKt_softmax = compute_softmax (scores,i)



[[0.30099825 0.36547517 0.14192817 0.19159841]
 [0.3039025  0.38305201 0.13117518 0.18187031]
 [0.27735478 0.30949121 0.19231494 0.22083907]
 [0.28587683 0.32485883 0.17287512 0.21638922]]


#### Solution to question 3 (i=3, j=2)

$i$ is the query vector position and $j$ is the key vector position.


#### Scaled dot product  (= scores) validation

In [110]:
# Look up requested value in the `scores` matrix defined in Question 2

This should equal the computation using Equation (8.12M) above.

In [113]:
# computations here

#### Question 5

In [376]:
def compute_head(QKt_soft,i, Xv):
   # Code goes here

# We do i = 3

i=3
head_i = compute_head(QKt_soft,i,Xv)
a_i = ?

#### Question 6

Computation of a_i for i=5  goes here

Is the second $a_{i}$ computation the same as the first?   

[answer here]

#### Question 7

#### Put the definition of your multiheaded attention module in the code cell below

You should pass the appropriate shape tuple to 

```
np.random.rand( ...)
```

to create any new matrices (as was done in the Part One Attention Module cell).


In [ ]:
import numpy as np
import random

# Same words means `shape`, `N`, `X` are the same

####### The new weight matrices, the new Wo
np.random.seed(47)
k_shape = ?
q_shape = ?
v_shape = ?
o_shape = ?
Wk2 = ?  
Wq2 = ?   
Wv2 = ?
# Wo2 must accept concatenations of 2 value vectors
Wo2 =  ?  # use 

Xk2 = ?
Xq2 = ?
Xv2 = ?

QKt2 = ?
scores2 = ?
QKt_softmax2 = compute_softmax (scores2,i)
head2_i = compute_head(QKt_softmax2, i, Xv2)
multi_heads_i = ? # You can use head_i from Question 5 as your second head
a2_i = ?

## Part Two

A drawback of naively parallelizing an attention model as we did in part
one is that the attention scores from position $j$ include information
for positions following $j$ in the input string.  This is not part of
our original conception of the task of a language model: predicting the next word.
If we are given the identity of the next word as part of the information
we use to predict it, that makes any sucessful predictions we make far less impressive.
So in this part of the assignment we turn to `Causal Attention` as introduced in the
`building_a_large_language_model` notebook. In causal attention,
a mask is used so as to screen off information from word vectors later in
the input when computing attention scores.

Question 8 concerns the attention module defined below as a `torch` neural net.  The definition is a modified version of the `CausalAttention` nn defined in the `building_a_large_language_model` notebook.

In [117]:
import torch
from torch import nn

class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        ##  Not a parameter but needs to be moved along with parameters
        ##  when we move data to GPU
        self.register_buffer(
           'mask',
           torch.triu(torch.ones(context_length, context_length),
           diagonal=1)
        )             # New: call to `register_buffer` for upper tri mask

    def forward(self, x):
        b, context_length, d_in = x.shape    
        # keys is batch_size x context_length x d_out
        self.keys = self.W_key(x)
        self.queries = self.W_query(x)
        self.values = self.W_value(x)
        # keys.transpose(1, 2)  is batch_size x d_out x context_length
        # making it compatible with matrix multiplication by queries.
        # attn_scores is therefore batch_size x d_out x d_out
        self.attn_scores = self.queries @ self.keys.transpose(1, 2)   
        # PyTorch convention: operations with a trailing 
        # underscore are in-place.
        self.attn_scores.masked_fill_(                                                       
            self.mask.bool()[:context_length, :context_length], -torch.inf) 
        self.attn_weights = torch.softmax(
            self.attn_scores / self.keys.shape[-1]**0.5, dim=-1
        )
        self.context_vec = self.attn_weights @ self.values
        return self.context_vec


input_sentence = ['Your', 'journey', 'starts', 'with', 'one', 'step']
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (input_sentence[0])
   [0.55, 0.87, 0.66], # journey  (input_sentence[1])
   [0.57, 0.85, 0.64], # starts   (input_sentence[2])
   [0.22, 0.58, 0.33], # with     (input_sentence[3])
   [0.77, 0.25, 0.10], # one      (input_sentence[4])
   [0.05, 0.80, 0.55]] # step     (input_sentence[5])
)

inputs_batch = torch.stack((inputs, ), dim=0)

Here is the result of passing a batch with a batchsize of 1 through the `CausalAttention` NN.

In [118]:
d_in = inputs_batch.shape[2]  
d_out=2
context_length = inputs_batch.shape[1]
torch.manual_seed(789)
ca = CausalAttention(d_in, d_out,context_length=context_length)
outputs = ca(inputs_batch)
print(outputs)

tensor([[[-0.0872,  0.0286],
         [-0.0991,  0.0501],
         [-0.0999,  0.0633],
         [-0.0983,  0.0489],
         [-0.0514,  0.1098],
         [-0.0754,  0.0693]]], grad_fn=<UnsafeViewBackward0>)


The network has been modified so that intermediate steps in the attention
computation can be examined after an output has been computed. For example,
examining the definition of the `CasualAttention` `forward` method, we
see that one of the steps computes the value vectors and saves them in `self.values`.
Accordingly, the `CasualAttention` instance `ca` has the values attribute 
`values`, which we can inspect:

In [119]:
ca.values

tensor([[[-0.0872,  0.0286],
         [-0.1137,  0.0766],
         [-0.1018,  0.0927],
         [-0.0912, -0.0026],
         [ 0.1395,  0.3580],
         [-0.2085, -0.1546]]], grad_fn=<UnsafeViewBackward0>)

As in answering all questions, you must show how you computed the answers in order to receive any credit for answering question 8.

**Question 8** Assume that the `inputs` tensor defined in the same cell with `CausalAttention` corresponds to the input sentence: *Your journey starts with one step.*  Execute the last few code cells to compute tensor `outputs` by passing the `inputs` batch through the `CausalAttention` Neural Net.  Answer these questions. (a) Based on the attention weights after `outputs` has been computed, what word has the highest attention weight when the current word is *starts* (using the terminology of Part One, when $j$ corresponds to *starts$, what $i$ gets the highest attention weight)? (b) What word has the highest attention weight when the current word is *step*? You may find that `input_sentence` is useful in writing your code; (c) What attention weight does the word *step* receive when the current word is  *Your*? What is it about the `CausalAttention` model that makes the value so small? 

## Code answers to (8a)-(8c) in the code cell below

Anwer to the question **What is it about the CausalAttention model that makes the value so small?** here:
